# Actividad 2026 — Q-learning: resolver, intervenir y auditar

**Ampliación formativa optativa: sin nota ni entrega obligatoria.** El único TP del curso es Decision Transformer.

Adaptación original del [hands-on de Hugging Face](https://huggingface.co/learn/deep-rl-course/unit2/hands-on). El entorno y la evaluación están provistos; completar la actualización y el análisis. No se requiere IA, GPU ni cuenta en el Hub.

**Colab:** abrir o subir solamente este `.ipynb` y ejecutar las celdas en orden. La primera instala los paquetes que falten; las funciones de experimentación están incorporadas aquí. **Local:** Python 3.11–3.13; la primera celda instala los paquetes que falten. No instalar el entorno del repositorio en una Jetson.

**Estado inicial:** la celda de Q-learning está pendiente. «Run all» se detendrá allí hasta que la implementen.


In [ ]:
from pathlib import Path
import importlib.metadata
import importlib.util
import subprocess
import sys

packages = {"gymnasium": "gymnasium==1.2.1", "numpy": "numpy", "matplotlib": "matplotlib"}
missing = [spec for name, spec in packages.items() if importlib.util.find_spec(name) is None]
if importlib.util.find_spec("gymnasium") is not None and importlib.metadata.version("gymnasium") != "1.2.1":
    missing.append("gymnasium==1.2.1")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

candidates = [Path.cwd() / "2026/actividades/q_learning", Path.cwd() / "actividades/q_learning", Path.cwd()]
BASE = next((p for p in candidates if (p / "lab_1_q_learning_auditoria.ipynb").exists()), Path.cwd())
print("Gymnasium", gym.__version__, "NumPy", np.__version__, "· resultados en", BASE / "outputs")


In [ ]:
# Infraestructura incluida para que el notebook funcione sin archivos auxiliares.
"""Infraestructura de la actividad formativa de Q-learning. La actualización se inyecta desde el notebook."""
import json
import platform
from pathlib import Path

import gymnasium as gym
import numpy as np


def greedy(q_row, rng):
    """Desempate aleatorio: evita favorecer por accidente a la acción cero."""
    return int(rng.choice(np.flatnonzero(q_row == q_row.max())))


def make_env(env_id, slippery=False):
    if env_id == "FrozenLake-v1":
        return gym.make(env_id, is_slippery=slippery, max_episode_steps=100)
    if env_id == "Taxi-v3":
        return gym.make(env_id, max_episode_steps=200)
    raise ValueError("Esta actividad fija FrozenLake-v1 y Taxi-v3.")


def train(update, env_id, seed, steps=100000, slippery=False, bonus=0.0,
          alpha=0.3, gamma=0.95, min_epsilon=0.1):
    """Presupuesto exacto en transiciones, independiente del largo del episodio."""
    env = make_env(env_id, slippery)
    rng = np.random.default_rng(seed)
    q = np.zeros((env.observation_space.n, env.action_space.n))
    state, _ = env.reset(seed=seed)
    history, original, modified, length = [], 0.0, 0.0, 0
    for t in range(steps):
        epsilon = min_epsilon + (1 - min_epsilon) * np.exp(-6 * t / steps)
        action = int(rng.integers(env.action_space.n)) if rng.random() < epsilon else greedy(q[state], rng)
        next_state, reward, terminated, truncated, _ = env.step(action)
        shaped = reward + (bonus if not terminated else 0.0)
        q[state, action] = update(q, state, action, shaped, next_state,
                                  terminated, truncated, alpha, gamma)
        original += reward
        modified += shaped
        length += 1
        state = next_state
        if terminated or truncated or t == steps - 1:
            history.append({"steps": t + 1, "length": length,
                            "original_return": float(original),
                            "modified_return": float(modified),
                            "terminated": bool(terminated), "truncated": bool(truncated),
                            "budget_cut": bool(not (terminated or truncated))})
            state, _ = env.reset()  # Continuar el RNG del entorno; no resembrar cada episodio.
            original, modified, length = 0.0, 0.0, 0
    env.close()
    return q, history


def evaluate(env_id, seed, q=None, episodes=100, slippery=False, split="validation"):
    if split not in ("validation", "test"):
        raise ValueError("split debe ser validation o test")
    # Reset seeds reservadas, distintas entre particiones y de las usadas en train.
    offset = 1000000 if split == "validation" else 2000000
    rng = np.random.default_rng(offset + seed)
    env = make_env(env_id, slippery)
    records = []
    for episode in range(episodes):
        state, _ = env.reset(seed=offset + episode)
        total, length, success = 0.0, 0, False
        while True:
            action = int(rng.integers(env.action_space.n)) if q is None else greedy(q[state], rng)
            state, reward, terminated, truncated, _ = env.step(action)
            total += reward
            length += 1
            success = bool(terminated and reward > 0)
            if terminated or truncated:
                break
        records.append({"return": float(total), "length": length,
                        "success": success, "truncated": bool(truncated),
                        "reset_seed": offset + episode})
    env.close()
    return {"return": float(np.mean([r["return"] for r in records])),
            "success": float(np.mean([r["success"] for r in records])),
            "length": float(np.mean([r["length"] for r in records])),
            "truncated": float(np.mean([r["truncated"] for r in records])),
            "episodes": records}


def run_suite(update, steps=100000, seeds=(11, 22, 33, 44, 55), episodes=100,
              split="validation", output=None):
    scenarios = [("lake_determinista", "FrozenLake-v1", False, 0.0),
                 ("lake_resbaladizo", "FrozenLake-v1", True, 0.0),
                 ("taxi", "Taxi-v3", False, 0.0),
                 ("lake_bonus", "FrozenLake-v1", True, 0.05)]
    result = {"versions": {"python": platform.python_version(), "numpy": np.__version__,
                           "gymnasium": gym.__version__},
              "config": {"steps": steps, "seeds": list(seeds), "eval_episodes": episodes,
                         "split": split, "alpha": 0.3, "gamma": 0.95,
                         "epsilon": "0.1 + 0.9 * exp(-6*t/steps)"}, "scenarios": {}}
    for name, env_id, slippery, bonus in scenarios:
        runs = []
        for seed in seeds:
            q, history = train(update, env_id, seed, steps, slippery, bonus)
            score = evaluate(env_id, seed, q, episodes, slippery, split)
            random_score = evaluate(env_id, seed, None, episodes, slippery, split)
            runs.append({"seed": seed, "trained": score, "random": random_score,
                         "train_history": history})
            if output:
                Path(output).mkdir(parents=True, exist_ok=True)
                np.savez(Path(output) / (name + "_" + str(seed) + ".npz"), q=q)
        summary = {}
        for policy in ("trained", "random"):
            summary[policy] = {metric: {"mean": float(np.mean([r[policy][metric] for r in runs])),
                                       "seed_std": float(np.std([r[policy][metric] for r in runs], ddof=1))
                                       if len(runs) > 1 else None}
                               for metric in ("return", "success", "length", "truncated")}
        result["scenarios"][name] = {"env": env_id, "slippery": slippery,
                                     "training_bonus": bonus, "runs": runs, "summary": summary}
        print(name, json.dumps(summary, ensure_ascii=False))
    if output:
        (Path(output) / "resultados.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result


def numerical_checks(update):
    q = np.array([[0., 0., 0., 0.], [8., 1., 0., 0.]])
    assert np.isclose(update(q, 0, 0, 0., 1, False, False, 1., .9), 7.2), "max != argmax"
    q[1, 0] = 10.
    assert np.isclose(update(q, 0, 0, 1., 1, True, False, 1., .9), 1.), "No bootstrap terminal"
    assert np.isclose(update(q, 0, 0, 1., 1, False, True, 1., .9), 10.), "Bootstrap al truncar"
    print("Tres contraejemplos numéricos verificados.")


def trace_policy(q, seed=2000000, slippery=True):
    """Rollout FrozenLake sin aprendizaje ni shaping, útil para interpretar loops."""
    env = make_env("FrozenLake-v1", slippery)
    state, _ = env.reset(seed=seed)
    rng = np.random.default_rng(seed)
    trace = []
    while True:
        action = greedy(q[state], rng)
        next_state, reward, terminated, truncated, _ = env.step(action)
        trace.append({"state": int(state), "action": action, "next_state": int(next_state),
                      "reward": float(reward), "terminated": bool(terminated), "truncated": bool(truncated)})
        state = next_state
        if terminated or truncated:
            break
    env.close()
    return trace


## 1. Registrar predicciones

Antes de correr, anotar dos predicciones sobre el lago resbaladizo y el bonus por permanecer vivo. ¿Qué resultado las refutaría?


## 2. Actualización

Para un corte externo por tiempo: $r + \gamma(1-\mathbb{1}_{terminated})\max_{a'}Q(s',a')$.

La función debe **devolver el nuevo valor escalar**, sin modificar la tabla. `truncated` se entrega para que expliquen por qué no se usa como máscara terminal aquí.

In [ ]:
def q_update(q, state, action, reward, next_state, terminated, truncated, alpha, gamma):
    # TODO: implementar Bellman y devolver Q(s,a) actualizado.
    raise NotImplementedError("Completar la actualización de Q-learning")


In [ ]:
numerical_checks(q_update)


## 3. Corridas

`demo`: una semilla, 3.000 pasos, 30 episodios para verificar instalación. `completo` (ampliación optativa): cinco semillas, 100.000 pasos y 100 episodios por política/configuración. Seleccionar hiperparámetros con validación y usar test una vez después de fijarlos. Las métricas de evaluación usan recompensa original y no actualizan Q.

La configuración de entrenamiento está explícita en este notebook; si la cambian, declarar y registrar la modificación.

In [ ]:
MODE = "demo"  # Cambiar a "completo" para el experimento ampliado.
SPLIT = "validation"  # Usar "test" solo con decisiones ya fijadas.
steps = 3000 if MODE == "demo" else 100000
seeds = [11] if MODE == "demo" else [11, 22, 33, 44, 55]
episodes = 30 if MODE == "demo" else 100
results = run_suite(q_update, steps=steps, seeds=seeds, episodes=episodes,
                    split=SPLIT, output=BASE / "outputs" / (MODE + "_" + SPLIT))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name in ["lake_resbaladizo", "lake_bonus"]:
    history = results["scenarios"][name]["runs"][0]["train_history"]
    x = [h["steps"] for h in history]
    window = min(30, len(history))
    for ax, metric in zip(axes, ["original_return", "modified_return"]):
        y = np.convolve([h[metric] for h in history], np.ones(window)/window, mode="valid")
        ax.plot(x[window-1:], y, label=name)
        ax.set(xlabel="Pasos de interacción", ylabel=metric)
        ax.legend()
plt.suptitle("Primera semilla: ilustración; el informe debe comparar todas")
plt.tight_layout()
plt.show()


## 4. Auditar propuestas

Para cada propuesta, identificar la causa, construir un contraejemplo, corregir y verificar. No comparar solo curvas: pueden ocultar el defecto.

In [ ]:
def propuesta_a(q, s, a, r, ns, terminated, truncated, alpha, gamma):
    target = r + gamma * (not terminated) * np.argmax(q[ns])
    return q[s, a] + alpha * (target - q[s, a])

def propuesta_b(q, s, a, r, ns, terminated, truncated, alpha, gamma):
    target = r + gamma * (not (terminated or truncated)) * np.max(q[ns])
    return q[s, a] + alpha * (target - q[s, a])

# TODO: agregar aquí contraejemplos y versiones corregidas.


## 5. Una trayectoria para interpretar el bonus

Cargar una Q-table guardada de `lake_bonus`, usar `make_env`, `reset(seed=...)` y `greedy`. Registrar estado, acción, siguiente estado, retorno original y si termina/trunca. No entrenar durante ese rollout. Comparar con `lake_resbaladizo` usando el mismo reset seed.

In [ ]:
# TODO: registrar y comentar una trayectoria de cada política.


## 6. Cierre formativo

Compartir una observación y una decisión verificada, con o sin IA. Distinguir recompensa modificada, retorno original y éxito. ¿Cuánto depende la conclusión de la semilla? ¿Qué cambió realmente en la política? La hoja de trabajo es opcional y no se entrega.
